In [2]:
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import random
import re



def clean_line(line):
    line = line.lower()
    line = re.sub(r"[^a-z']", ' ', line)
    return line.split()

def make_ngrams(words, n):
    return [tuple(words[i:i+n]) for i in range(len(words)-n+1)]

def load_data(train_files, n):
    vocab = set()
    data_per_author = {}
    author_vocabs = {}
    for author, file in train_files.items():
        data_per_author[author] = []
        author_vocabs[author] = set()
        with open(file, 'r', encoding='utf-8') as f:
            for line in f:
                words = clean_line(line)
                author_vocabs[author].update(words)
                if len(words) < n:
                    continue
                for ng in make_ngrams(words, n):
                    data_per_author[author].append(ng)
                    vocab.update(ng)
    vocab = list(vocab) + ['OOV']
    wd2ix = {w:i for i,w in enumerate(vocab)}
    auth2ix = {a:i for i,a in enumerate(train_files)}
    return vocab, wd2ix, auth2ix, data_per_author, author_vocabs

def ngrams_from_test(file, n, wd2ix):
    ngrams = []
    with open(file, 'r', encoding='utf-8') as f:
        for line in f:
            words = clean_line(line)
            if len(words) < n:
                continue
            ngs = make_ngrams(words, n)
            ngs = [tuple([w if w in wd2ix else 'OOV' for w in ng]) for ng in ngs]
            ngrams.extend(ngs)
    return ngrams

class AuthorNet(nn.Module):
    def __init__(self, vocab_size, emb_dim, ngram_n, num_auths, hidden_dim=128):
        super().__init__()
        self.embeddings = nn.Embedding(vocab_size, emb_dim)
        self.hidden = nn.Linear(emb_dim * ngram_n, hidden_dim)
        self.relu = nn.ReLU()
        self.linear = nn.Linear(hidden_dim, num_auths)
        self.softmax = nn.Softmax(dim=1)
    def forward(self, x):
        embs = [self.embeddings(x[:, i]) for i in range(x.shape[1])]
        concat_emb = torch.cat(embs, dim=1)
        hidden_out = self.relu(self.hidden(concat_emb))
        out = self.linear(hidden_out)
        return out

def get_batch(data_per_author, wd2ix, auth2ix, ngram_n, batch_size=32):
    X, y = [], []
    authors = list(data_per_author.keys())
    for _ in range(batch_size):
        a = random.choice(authors)
        ngram = random.choice(data_per_author[a])
        X.append([wd2ix.get(w, wd2ix['OOV']) for w in ngram])
        y.append(auth2ix[a])
    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

def main(train_files, test_files, n=4, emb_dim=32, lr=0.003, hidden_dim=128):
    if len(sys.argv) == 1 or len(sys.argv) > 2:
        num_iter = 20000
    else:
        num_iter = int(sys.argv[1])

    vocab, wd2ix, auth2ix, data_per_author, author_vocabs = load_data(train_files, n)
    num_authors = len(auth2ix)
    vocab_size = len(vocab)

    print("Common counts:")
    all_vocabs = [author_vocabs[a] for a in train_files]
    total_vocab_all = set().union(*all_vocabs)
    for author in train_files:
        total_ngrams = len(data_per_author[author])
        common_count = sum(1 for ng in data_per_author[author] if all(w in set.intersection(*all_vocabs) for w in ng))
        ratio = common_count / total_ngrams if total_ngrams > 0 else 0
        print(f"{author}_train.txt {common_count} {total_ngrams} {ratio:.4f}")
    print(f"The total vocab for all the books is {len(total_vocab_all)}\n")

    model = AuthorNet(vocab_size, emb_dim, n, num_authors, hidden_dim)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=lr/10)
    loss_fn = nn.CrossEntropyLoss()

    batch_size = 8
    correct, sample_count, losses = 0, 0, []

    for it in range(1, num_iter + 1):
        X, y = get_batch(data_per_author, wd2ix, auth2ix, n, batch_size)
        optimizer.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, y)
        losses.append(loss.item())
        loss.backward()
        optimizer.step()

        correct += (pred.argmax(dim=1) == y).sum().item()
        sample_count += batch_size

        if sample_count >= 1000:
            avg_loss = sum(losses[-(sample_count):]) / sample_count
            print(f"{correct} out of 1000 correct; avg ep loss on last 1000 {avg_loss:.4f}")
            correct = 0
            sample_count = 0

    print("Now testing.\n")

    model.eval()
    with torch.no_grad():
        for test_author, file in test_files.items():
            ngrams = ngrams_from_test(file, n, wd2ix)
            agg_logprobs = torch.zeros(num_authors)
            for ng in ngrams:
                x = torch.tensor([[wd2ix.get(w, wd2ix['OOV']) for w in ng]])
                pred = model(x)
                log_probs = torch.log(model.softmax(pred))
                agg_logprobs += log_probs[0]
            pred_idx = torch.argmax(agg_logprobs).item()
            log_line = f"{test_author} " + " ".join([f"{v:.4f}" for v in agg_logprobs.tolist()])
            log_line += f" Best score index: {pred_idx}"
            print(log_line)
            print(f"which is {list(auth2ix.keys())[pred_idx]}\n")

if __name__ == "__main__":
    train_files = {
        "Coleridge": "Coleridge_train.txt",
        "Eliot": "Eliot_train.txt",
        "EBBrowning": "EBBrowning_train.txt",
        "Emily Dickinson": "Emily_Dickinson_train.txt",
        "Shakespeare": "Shakespeares_sonnets_train.txt"
    }
    test_files = {
        "Coleridge": "Coleridge_test.txt",
        "Eliot": "Eliot_test.txt",
        "EBBrowning": "EBBrowning_test.txt",
        "Emily Dickinson": "Emily_Dickinson_test.txt",
        "Shakespeare": "Shakespeares_sonnets_test.txt"
    }
    main(train_files, test_files)




Common counts:
Coleridge_train.txt 126 2618 0.0481
Eliot_train.txt 178 2446 0.0728
EBBrowning_train.txt 147 2638 0.0557
Emily Dickinson_train.txt 115 1740 0.0661
Shakespeare_train.txt 193 2789 0.0692
The total vocab for all the books is 4724

247 out of 1000 correct; avg ep loss on last 1000 0.2015
276 out of 1000 correct; avg ep loss on last 1000 0.3987
328 out of 1000 correct; avg ep loss on last 1000 0.5920
350 out of 1000 correct; avg ep loss on last 1000 0.7801
367 out of 1000 correct; avg ep loss on last 1000 0.9622
413 out of 1000 correct; avg ep loss on last 1000 1.1384
433 out of 1000 correct; avg ep loss on last 1000 1.3112
439 out of 1000 correct; avg ep loss on last 1000 1.4809
505 out of 1000 correct; avg ep loss on last 1000 1.4378
517 out of 1000 correct; avg ep loss on last 1000 1.3929
563 out of 1000 correct; avg ep loss on last 1000 1.3430
580 out of 1000 correct; avg ep loss on last 1000 1.2927
644 out of 1000 correct; avg ep loss on last 1000 1.2347
606 out of 1000 